In [1]:
import pandas as pd
import requests
import numpy as np
from datetime import datetime
from meteostat import Point, Hourly
import time
from functools import lru_cache
import holidays

In [2]:
df = pd.read_csv('historique_stations.csv', 
                 names=['date', 'capacity', 'mechanic_available', 
                        'electrical_available', 'station_name', 
                        'station_position', 'operative'])
df = df[df['operative'] == True].copy()
df

,date,capacity,mechanic_available,electrical_available,station_name,station_position,operative
0,2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
1,2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True
2,2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True
3,2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True
4,2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True
...,...,...,...,...,...,...,...
10986725,2021-04-09T14:37Z,38,4,2,Général Michel Bizot - Claude Decaen,"48.83481,2.40093",True
10986726,2021-04-09T14:37Z,20,2,1,Ivry - Baudricourt,"48.82470,2.36311",True
10986727,2021-04-09T14:37Z,39,17,0,Saint-Mandé - Docteur Arnold Netter,"48.84463,2.40495",True
10986728,2021-04-09T14:37Z,21,12,4,Saint-Marcel - Hôpital,"48.83950,2.36099",True


In [ ]:
#  Parse datetime features - REMOVE TIMEZONE to avoid conflicts
df['datetime'] = pd.to_datetime(df['date']).dt.tz_localize(None)
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['year'] = df['datetime'].dt.year
df['weekday'] = df['datetime'].dt.weekday

# Additional temporal features for better analysis
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df['day_of_year'] = df['datetime'].dt.dayofyear
df['week_of_year'] = df['datetime'].dt.isocalendar().week

def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'

df['season'] = df['month'].apply(get_season)

def add_holiday_features(df):
    years = df['year'].unique().tolist()    
    fr_holidays = holidays.France(years=years)

    sample_year = years[0] if len(years) > 0 else None
    if sample_year:
        year_holidays = {k: v for k, v in fr_holidays.items() if k.year == sample_year}
        for date, name in list(year_holidays.items())[:5]:
            print(f"  {date}: {name}")
    
    df['date_only'] = pd.to_datetime(df['datetime']).dt.date
    
    holiday_dates = set(fr_holidays.keys())
    
    df['is_public_holiday'] = df['date_only'].isin(holiday_dates).astype(int)
        
    # Check previous and next day holidays
    df['prev_date'] = df['date_only'] - pd.Timedelta(days=1)
    df['next_date'] = df['date_only'] + pd.Timedelta(days=1)
    
    df['prev_day_holiday'] = df['prev_date'].isin(holiday_dates).astype(int)
    df['next_day_holiday'] = df['next_date'].isin(holiday_dates).astype(int)
    
    # Drop temporary columns
    df.drop(['date_only', 'prev_date', 'next_date'], axis=1, inplace=True)
    
    print(f"Found {df['is_public_holiday'].sum()} records on public holidays")
        
    return df

df = add_holiday_features(df)
df

  2020-01-01: Jour de l'an
  2020-04-13: Lundi de Pâques
  2020-06-01: Lundi de Pentecôte
  2020-05-01: Fête du Travail
  2020-05-08: Fête de la Victoire
Found 243993 records on public holidays


,date,capacity,mechanic_available,electrical_available,station_name,station_position,operative,datetime,hour,day,...,year,weekday,is_weekend,is_rush_hour,day_of_year,week_of_year,season,is_public_holiday,prev_day_holiday,next_day_holiday
0,2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True,2020-11-26 12:59:00,12,26,...,2020,3,0,0,331,48,3,0,0,0
1,2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True,2020-11-26 12:59:00,12,26,...,2020,3,0,0,331,48,3,0,0,0
2,2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True,2020-11-26 12:59:00,12,26,...,2020,3,0,0,331,48,3,0,0,0
3,2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True,2020-11-26 12:59:00,12,26,...,2020,3,0,0,331,48,3,0,0,0
4,2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True,2020-11-26 12:59:00,12,26,...,2020,3,0,0,331,48,3,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10986725,2021-04-09T14:37Z,38,4,2,Général Michel Bizot - Claude Decaen,"48.83481,2.40093",True,2021-04-09 14:37:00,14,9,...,2021,4,0,0,99,14,1,0,0,0
10986726,2021-04-09T14:37Z,20,2,1,Ivry - Baudricourt,"48.82470,2.36311",True,2021-04-09 14:37:00,14,9,...,2021,4,0,0,99,14,1,0,0,0
10986727,2021-04-09T14:37Z,39,17,0,Saint-Mandé - Docteur Arnold Netter,"48.84463,2.40495",True,2021-04-09 14:37:00,14,9,...,2021,4,0,0,99,14,1,0,0,0
10986728,2021-04-09T14:37Z,21,12,4,Saint-Marcel - Hôpital,"48.83950,2.36099",True,2021-04-09 14:37:00,14,9,...,2021,4,0,0,99,14,1,0,0,0


In [5]:
# ADD ZIPCODE
@lru_cache(maxsize=1000)
def get_postal_code(lat, lon):
    url = f"https://api-adresse.data.gouv.fr/reverse/?lat={lat}&lon={lon}"
    try:
        time.sleep(0.1)  # Rate limiting: 10 requests per second max
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if data.get('features') and len(data['features']) > 0:
            postal_code = data['features'][0]['properties'].get('postcode')
            return postal_code if postal_code else 'Unknown'
        return 'Unknown'
    except requests.exceptions.RequestException as e:
        print(f"Error fetching postal code for {lat}, {lon}: {e}")
        return 'Unknown'
    except (KeyError, IndexError) as e:
        print(f"Error parsing response for {lat}, {lon}: {e}")
        return 'Unknown'

unique_station_geo = df['station_position'].unique()
station_to_zipcode = {}

for idx, station_pos in enumerate(unique_station_geo):
    if pd.isna(station_pos):
        station_to_zipcode[station_pos] = 'Unknown'
        continue
    
    try:
        lat, lon = station_pos.split(",")
        lat, lon = lat.strip(), lon.strip()
        postal_code = get_postal_code(float(lat), float(lon))
        station_to_zipcode[station_pos] = postal_code
    except Exception as e:
        print(f"Error processing station {station_pos}: {e}")
        station_to_zipcode[station_pos] = 'Unknown'
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(unique_station_geo)} stations")

df['zipcode'] = df['station_position'].map(station_to_zipcode)

Processed 100/1384 stations
Processed 200/1384 stations
Processed 300/1384 stations
Processed 400/1384 stations
Processed 500/1384 stations
Processed 600/1384 stations
Processed 700/1384 stations
Processed 800/1384 stations
Processed 900/1384 stations
Processed 1000/1384 stations
Processed 1100/1384 stations
Processed 1200/1384 stations
Processed 1300/1384 stations


In [6]:
def get_arrondissement(zipcode):
    if pd.isna(zipcode) or zipcode == 'Unknown':
        return 'Unknown'
    if str(zipcode).startswith('75') and len(str(zipcode)) == 5:
        return str(zipcode)[3:]  
    return '00' # Banlieue

df['arrondissement'] = df['zipcode'].apply(get_arrondissement)

In [ ]:
def get_weather_information(start_datetime, end_datetime):
    location = Point(48.8566, 2.3522, 35)  # Paris Coordinates
    
    try:
        # Ensure datetimes are timezone-naive for meteostat
        if hasattr(start_datetime, 'tz') and start_datetime.tz is not None:
            start_datetime = start_datetime.tz_localize(None)
        if hasattr(end_datetime, 'tz') and end_datetime.tz is not None:
            end_datetime = end_datetime.tz_localize(None)
        
        data = Hourly(location, start_datetime, end_datetime)
        data = data.fetch()
        
        if data.empty:
            print("Warning: No weather data retrieved")
            return None
        
        # Reset index to make datetime a column
        data = data.reset_index()
        data.rename(columns={'time': 'weather_datetime'}, inplace=True)
        
        # Ensure weather datetime is also timezone-naive
        data['weather_datetime'] = pd.to_datetime(data['weather_datetime']).dt.tz_localize(None)
        
        return data
    except Exception as e:
        print(f"Error fetching weather data: {e}")
        import traceback
        traceback.print_exc()
        return None
    
min_date = df['datetime'].min()
max_date = df['datetime'].max()

df_meteo = get_weather_information(min_date, max_date)

df['datetime_rounded'] = df['datetime'].dt.floor('h')
df_meteo['datetime_rounded'] = pd.to_datetime(df_meteo['weather_datetime']).dt.floor('h')

df_merged = pd.merge(df, df_meteo, 
                        left_on='datetime_rounded', 
                        right_on='datetime_rounded', 
                        how='left')

df_merged.drop(['datetime_rounded', 'weather_datetime'], axis=1, inplace=True)

weather_cols = ['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']
available_weather_cols = [col for col in weather_cols if col in df_merged.columns]

for col in available_weather_cols:
    df_merged[col] = df_merged[col].interpolate(method='linear', limit_direction='both')

if 'temp' in df_merged.columns:
    df_merged['temp_category'] = pd.cut(df_merged['temp'], 
                                            bins=[-float('inf'), 0, 10, 20, float('inf')],
                                            labels=['freezing', 'cold', 'mild', 'warm'])

if 'prcp' in df_merged.columns:
    df_merged['is_raining'] = (df_merged['prcp'] > 0).astype(int)

if 'wspd' in df_merged.columns:
    df_merged['is_windy'] = (df_merged['wspd'] > 20).astype(int)  # > 20 km/h

print(f"Final merged dataset shape: {df_merged.shape}")
print(f"Missing values per column:\n{df_merged.isnull().sum()}")

df_merged['total_bikes'] = df_merged['mechanic_available'] + df_merged['electrical_available']
df_merged['availability_rate'] = df_merged['total_bikes'] / df_merged['capacity']
df_merged['occupancy_rate'] = 1 - df_merged['availability_rate']

df_merged

Final merged dataset shape: (10769264, 37)
Missing values per column:
date                           0
capacity                       0
mechanic_available             0
electrical_available           0
station_name                   0
station_position               0
operative                      0
datetime                       0
hour                           0
day                            0
month                          0
year                           0
weekday                        0
is_weekend                     0
is_rush_hour                   0
day_of_year                    0
week_of_year                   0
season                         0
is_public_holiday              0
prev_day_holiday               0
next_day_holiday               0
zipcode                        0
arrondissement                 0
temp                           0
dwpt                           0
rhum                           0
prcp                           0
snow                           0
wdir  

,date,capacity,mechanic_available,electrical_available,station_name,station_position,operative,datetime,hour,day,...,wpgt,pres,tsun,coco,temp_category,is_raining,is_windy,total_bikes,availability_rate,occupancy_rate
0,2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True,2020-11-26 12:59:00,12,26,...,13.0,1019.6,<NA>,<NA>,mild,0,0,9,0.257143,0.742857
1,2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True,2020-11-26 12:59:00,12,26,...,13.0,1019.6,<NA>,<NA>,mild,0,0,27,0.490909,0.509091
2,2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True,2020-11-26 12:59:00,12,26,...,13.0,1019.6,<NA>,<NA>,mild,0,0,0,0.000000,1.000000
3,2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True,2020-11-26 12:59:00,12,26,...,13.0,1019.6,<NA>,<NA>,mild,0,0,1,0.047619,0.952381
4,2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True,2020-11-26 12:59:00,12,26,...,13.0,1019.6,<NA>,<NA>,mild,0,0,4,0.133333,0.866667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10769259,2021-04-09T14:37Z,38,4,2,Général Michel Bizot - Claude Decaen,"48.83481,2.40093",True,2021-04-09 14:37:00,14,9,...,35.2,1012.9,<NA>,3.0,mild,0,0,6,0.157895,0.842105
10769260,2021-04-09T14:37Z,20,2,1,Ivry - Baudricourt,"48.82470,2.36311",True,2021-04-09 14:37:00,14,9,...,35.2,1012.9,<NA>,3.0,mild,0,0,3,0.150000,0.850000
10769261,2021-04-09T14:37Z,39,17,0,Saint-Mandé - Docteur Arnold Netter,"48.84463,2.40495",True,2021-04-09 14:37:00,14,9,...,35.2,1012.9,<NA>,3.0,mild,0,0,17,0.435897,0.564103
10769262,2021-04-09T14:37Z,21,12,4,Saint-Marcel - Hôpital,"48.83950,2.36099",True,2021-04-09 14:37:00,14,9,...,35.2,1012.9,<NA>,3.0,mild,0,0,16,0.761905,0.238095


In [8]:
df_merged.drop(['tsun', 'coco'], axis=1, inplace=True)
df_merged

,date,capacity,mechanic_available,electrical_available,station_name,station_position,operative,datetime,hour,day,...,wdir,wspd,wpgt,pres,temp_category,is_raining,is_windy,total_bikes,availability_rate,occupancy_rate
0,2020-11-26T12:59Z,35,4,5,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True,2020-11-26 12:59:00,12,26,...,120.0,3.6,13.0,1019.6,mild,0,0,9,0.257143,0.742857
1,2020-11-26T12:59Z,55,23,4,André Mazet - Saint-André des Arts,"48.85376,2.33910",True,2020-11-26 12:59:00,12,26,...,120.0,3.6,13.0,1019.6,mild,0,0,27,0.490909,0.509091
2,2020-11-26T12:59Z,20,0,0,Charonne - Robert et Sonia Delauney,"48.85591,2.39257",True,2020-11-26 12:59:00,12,26,...,120.0,3.6,13.0,1019.6,mild,0,0,0,0.000000,1.000000
3,2020-11-26T12:59Z,21,0,1,Toudouze - Clauzel,"48.87930,2.33736",True,2020-11-26 12:59:00,12,26,...,120.0,3.6,13.0,1019.6,mild,0,0,1,0.047619,0.952381
4,2020-11-26T12:59Z,30,3,1,Mairie du 12ème,"48.84086,2.38755",True,2020-11-26 12:59:00,12,26,...,120.0,3.6,13.0,1019.6,mild,0,0,4,0.133333,0.866667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10769259,2021-04-09T14:37Z,38,4,2,Général Michel Bizot - Claude Decaen,"48.83481,2.40093",True,2021-04-09 14:37:00,14,9,...,210.0,16.6,35.2,1012.9,mild,0,0,6,0.157895,0.842105
10769260,2021-04-09T14:37Z,20,2,1,Ivry - Baudricourt,"48.82470,2.36311",True,2021-04-09 14:37:00,14,9,...,210.0,16.6,35.2,1012.9,mild,0,0,3,0.150000,0.850000
10769261,2021-04-09T14:37Z,39,17,0,Saint-Mandé - Docteur Arnold Netter,"48.84463,2.40495",True,2021-04-09 14:37:00,14,9,...,210.0,16.6,35.2,1012.9,mild,0,0,17,0.435897,0.564103
10769262,2021-04-09T14:37Z,21,12,4,Saint-Marcel - Hôpital,"48.83950,2.36099",True,2021-04-09 14:37:00,14,9,...,210.0,16.6,35.2,1012.9,mild,0,0,16,0.761905,0.238095


In [ ]:
# output_file = 'velib_augmented_dataset.csv'
# df_merged.to_csv(output_file, index=False)

In [11]:
print(df_merged.columns.tolist())

['date', 'capacity', 'mechanic_available', 'electrical_available', 'station_name', 'station_position', 'operative', 'datetime', 'hour', 'day', 'month', 'year', 'weekday', 'is_weekend', 'is_rush_hour', 'day_of_year', 'week_of_year', 'season', 'is_public_holiday', 'prev_day_holiday', 'next_day_holiday', 'zipcode', 'arrondissement', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'temp_category', 'is_raining', 'is_windy', 'total_bikes', 'availability_rate', 'occupancy_rate']
